# Integracao: LoadBalancer + Logger + Metrics

Este notebook demonstra uma execucao simples da simulacao, integrando:
- `LoadBalancer` para roteamento
- `MetricsCollector` para eventos e metricas
- `SimulationLogger` para logs estruturados

In [1]:
import io
import logging
import os
import sys

import simpy

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'lab1':
    ROOT = os.path.abspath(os.getcwd())

SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from load_balancer_sim.config import SimulationConfig
from load_balancer_sim.load_balancer import LoadBalancer
from load_balancer_sim.logs import LOGGER_NAME, SimulationLogger
from load_balancer_sim.metrics import MetricsCollector
from load_balancer_sim.request import Request
from load_balancer_sim.server import Server

In [2]:
# Configura logger para capturar saida em memoria e exibir no notebook.
log_buffer = io.StringIO()
stream_handler = logging.StreamHandler(log_buffer)
stream_handler.setFormatter(logging.Formatter('%(levelname)s %(message)s'))

logger = logging.getLogger(LOGGER_NAME)
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
logger.addHandler(stream_handler)
logger.propagate = False

sim_logger = SimulationLogger(logger=logger)

In [3]:
# Instanciacao do cenario da simulacao.
config = SimulationConfig(
    policy='round_robin',
    server_count=3,
    server_capacity=2,
    service_time=0.05,
    horizon=1.0,
)

environment = simpy.Environment()
servers = [
    Server(
        environment,
        server_id=index,
        capacity=config.server_capacity,
        service_time=config.service_time,
    )
    for index in range(config.server_count)
]

load_balancer = LoadBalancer(
    environment=environment,
    servers=servers,
    policy=config.policy,
    simulation_config=config,
)

collector = MetricsCollector(environment, event_handler=sim_logger.log_event)
sim_logger.log_run_started(config)

In [4]:
def track_request_lifecycle(env, request, server, metrics_collector):
    """Registra inicio e conclusao no instante em que ocorrerem."""
    while request.service_start_time is None:
        yield env.timeout(1e-6)
    metrics_collector.record_service_started(request, server)

    while request.completion_time is None:
        yield env.timeout(1e-6)
    metrics_collector.record_service_completed(request, server)


def dispatch_request(env, lb, metrics_collector, request_id, burst_id=0):
    request = Request(id=request_id, burst_id=burst_id, arrival_time=env.now)
    metrics_collector.record_arrival(request)

    server = lb.route_request(request)
    metrics_collector.record_routing(request, server)

    env.process(track_request_lifecycle(env, request, server, metrics_collector))
    return request, server

In [5]:
# Envia algumas requisicoes quase simultaneas para observar roteamento e filas.
requests = []
for req_id in range(8):
    request, server = dispatch_request(environment, load_balancer, collector, req_id)
    requests.append((request, server.id))

# Executa a simulacao ate drenar os eventos agendados.
environment.run()

run_metrics = collector.calculate_run_metrics(horizon=config.horizon)
sim_logger.log_run_completed(run_metrics)

In [6]:
print('Resumo de metricas:')
print(run_metrics)

print('\nPrimeiros 12 eventos metricos:')
for event in collector.events[:12]:
    print(event)

print('\nEstado final dos servidores:')
for server in servers:
    print({
        'server_id': server.id,
        'completed_count': server.completed_count,
        'max_active': server.maximum_active_count,
        'max_waiting': server.maximum_waiting_count,
    })

Resumo de metricas:
RunMetrics(horizon=1.0, arrival_count=8, completed_count=8, pending_count=0, throughput=8.0, average_queue_time=0.012500750000007338, average_response_time=0.06250000000004184)

Primeiros 12 eventos metricos:
MetricEvent(time=0.0, event='arrival', request_id=0, burst_id=0, server_id=None, active_count=None, waiting_count=None)
MetricEvent(time=0.0, event='routing', request_id=0, burst_id=0, server_id=0, active_count=0, waiting_count=0)
MetricEvent(time=0.0, event='arrival', request_id=1, burst_id=0, server_id=None, active_count=None, waiting_count=None)
MetricEvent(time=0.0, event='routing', request_id=1, burst_id=0, server_id=1, active_count=0, waiting_count=0)
MetricEvent(time=0.0, event='arrival', request_id=2, burst_id=0, server_id=None, active_count=None, waiting_count=None)
MetricEvent(time=0.0, event='routing', request_id=2, burst_id=0, server_id=2, active_count=0, waiting_count=0)
MetricEvent(time=0.0, event='arrival', request_id=3, burst_id=0, server_id=Non

In [7]:
print('Saida de logs estruturados:')
print(log_buffer.getvalue())

Saida de logs estruturados:
INFO run_started,policy=round_robin,server_count=3,server_capacity=2,service_time=0.050000,burst_max=30,hurst=0.800000,horizon=1.000000,seed=12345
DEBUG time,event,request_id,burst_id,server_id,active,queue_length
DEBUG 0.000000,arrival,0,0,,,
DEBUG 0.000000,routing,0,0,0,0,0
DEBUG 0.000000,arrival,1,0,,,
DEBUG 0.000000,routing,1,0,1,0,0
DEBUG 0.000000,arrival,2,0,,,
DEBUG 0.000000,routing,2,0,2,0,0
DEBUG 0.000000,arrival,3,0,,,
DEBUG 0.000000,routing,3,0,0,0,0
DEBUG 0.000000,arrival,4,0,,,
DEBUG 0.000000,routing,4,0,1,0,0
DEBUG 0.000000,arrival,5,0,,,
DEBUG 0.000000,routing,5,0,2,0,0
DEBUG 0.000000,arrival,6,0,,,
DEBUG 0.000000,routing,6,0,0,0,0
DEBUG 0.000000,arrival,7,0,,,
DEBUG 0.000000,routing,7,0,1,0,0
DEBUG 0.000001,service_started,0,0,0,2,1
DEBUG 0.000001,service_started,1,0,1,2,1
DEBUG 0.000001,service_started,2,0,2,2,0
DEBUG 0.000001,service_started,3,0,0,2,1
DEBUG 0.000001,service_started,4,0,1,2,1
DEBUG 0.000001,service_started,5,0,2,2,0
DEBUG 0.